# 🤖 AI & Margin: When Does AI Help and When Does It Hurt?
**ML Project SEA – A.Y. 2025/2026 | Alkemy AI Productivity**

> **Research Question:** Beyond which AI usage threshold does rework destroy profit margin?

---

## Pipeline Overview

| Phase | Section | Purpose |
|-------|---------|---------|
| **Data Foundation** | §1–4 | Load, clean, engineer features |
| **Exploration** | §5 | Understand distributions and relationships |
| **Baseline Test** | §6 | AI vs No-AI comparison with statistical tests |
| **Core Analysis** | §7–9 | Non-linear effects, threshold detection |
| **Deep Dive** | §10–11 | Hidden costs, causal mechanism |
| **Validation** | §12 | Robustness across segments |
| **Modeling** | §13–14 | OLS regression, mediation, interaction effects |
| **Decision** | §15 | Business recommendation |
| **Reflection** | §16 | AI usage log, insights, mistakes |

## 1. Imports & Setup

We start by loading all the libraries we will need throughout the notebook. A brief overview of why each is needed:

- **pandas / numpy** — data manipulation and numerical computation
- **matplotlib / seaborn** — static visualizations with a custom dark-mode style
- **missingno** — specialized library for visualizing missing data patterns
- **scipy.stats** — statistical tests (Welch t-test, Pearson/Spearman correlations)
- **statsmodels** — LOWESS smoothing and OLS regression for mechanism analysis

We also define a consistent color palette so all charts in the notebook look cohesive.

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
from scipy import stats
import statsmodels.api as sm

warnings.filterwarnings('ignore')

# ── Visual Style ──────────────────────────────────────────────────────────────
# We use a dark-mode theme inspired by modern data dashboards.
# All charts will share this style for a professional, cohesive look.
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3d4a',
    'axes.labelcolor': '#e0e0e0',
    'axes.titlecolor': '#ffffff',
    'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0',
    'text.color': '#e0e0e0',
    'grid.color': '#2a2d3a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#3a3d4a',
    'font.family': 'sans-serif',
})

# Named palette for reuse across all visualizations
PALETTE = ['#7c6ae6', '#e06c75', '#56b6c2', '#e5c07b', '#98c379', '#d19a66']
SEA_BLUE   = '#7c6ae6'
SEA_RED    = '#e06c75'
SEA_GREEN  = '#98c379'
SEA_YELLOW = '#e5c07b'
SEA_CYAN   = '#56b6c2'

# ── File Paths ────────────────────────────────────────────────────────────────
HERE     = os.path.abspath(os.path.dirname('')) if '__file__' not in dir() else os.path.dirname(__file__)
DATA_DIR = os.path.join(HERE, 'data')
IMG_DIR  = os.path.join(HERE, 'images')
PATH     = os.path.join(DATA_DIR, 'ai_productivity_dataset_final.csv')

os.makedirs(IMG_DIR, exist_ok=True)

print('Data path:', PATH)
print('Images will be saved to:', IMG_DIR)
print('Dependencies loaded ✓')

## 2. Data Loading & First Look

Before doing anything, we need to understand what we're working with. We load the CSV and inspect:
- **Shape**: how many tasks (rows) and variables (columns)?
- **Columns**: what information is available for each task?
- **Head**: what do a few sample rows look like?

The dataset represents individual tasks/deliverables at Alkemy, each with information about hours worked, AI usage, quality scores, rework, and financial outcomes (revenue, cost, profit).

In [ ]:
df_raw = pd.read_csv(PATH)
print(f"Dataset shape: {df_raw.shape[0]} tasks × {df_raw.shape[1]} variables")
print(f"\nColumn names:\n{list(df_raw.columns)}")
df_raw.head(3)

Let's also check the data types and basic statistics. This helps us understand:
- Which columns are numeric vs. categorical vs. dates (stored as strings)
- The range and distribution of key numeric variables (min, max, mean, std)
- Whether any obvious issues jump out (e.g., negative hours, impossible percentages)

In [ ]:
print("── Data Types ──")
print(df_raw.dtypes.to_string())
print(f"\n── Numeric Summary ──")
df_raw.describe().round(2)

### 2b. Missing Value Overview

Before cleaning, we want to know *how much* data is missing and *where*. This informs our imputation strategy: if a critical variable like `ai_usage_pct` is missing for 4% of rows, that's manageable with median imputation. If it were 30%, we'd need a different approach.

We visualize the missing patterns using `missingno.matrix` — each white stripe is a missing value, so we can spot whether missingness is random or concentrated in specific rows/columns.

In [ ]:
# Count missing values per column (only columns with at least 1 NaN)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'n_missing': missing, 'pct': missing_pct})
print(missing_df[missing_df['n_missing'] > 0].sort_values('pct', ascending=False).to_string())

# Visual overview
fig, ax = plt.subplots(figsize=(12, 4))
msno.matrix(df_raw, ax=ax, color=(0.49, 0.42, 0.90), fontsize=9)
ax.set_title("Missing Value Matrix — Raw Dataset", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'fig1_missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n→ Figure saved to images/fig1_missing_values.png")

## 3. Data Cleaning

The project brief warns that the dataset is *"multi-table, incomplete, and imperfect."* We apply the following cleaning steps in order:

1. **Deduplication** — Remove rows with the same `task_id` (each task should appear only once)
2. **Team normalization** — Fix typos and case inconsistencies (e.g., `Contennt` → `Content`, `Paid Media` → `Media`)
3. **Date parsing** — Convert string date columns to proper datetime format for time-based analysis
4. **Legacy AI flag** — Map text values (`'true'`/`'false'`/`'unknown'`) to proper boolean/NaN
5. **Missing value imputation** — Fill numerical NaN with median; estimate missing `billable_hours`
6. **Outlier capping** — Use the IQR × 3 method to cap extreme values that would distort our analysis

> **Why median imputation?** The median is robust to skewed distributions (unlike the mean). Since financial data is typically right-skewed, using the mean would overestimate missing values.

### 3.1 Deduplication

If the same task appears twice, it will be counted twice in our aggregations — inflating counts and biasing averages. We drop duplicates based on `task_id`, keeping the first occurrence.

In [ ]:
df = df_raw.copy()

before = len(df)
df = df.drop_duplicates(subset='task_id')
after = len(df)
print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Duplicates removed: {before - after}")

### 3.2 Team Normalization

The `team` column contains typos and inconsistent casing. For example, `'Contennt'` should be `'Content'` and `'Paid Media'` should be mapped to `'Media'`. If we don't fix this, our team-level segmentation will have phantom groups that split the data incorrectly.

In [ ]:
print("Teams BEFORE normalization:", sorted(df['team'].unique()))

team_map = {
    'seo': 'SEO', 'SEO ': 'SEO',
    'media': 'Media', 'MEDIA': 'Media', 'Paid Media': 'Media',
    'content': 'Content', 'CONTENT': 'Content', 'Contennt': 'Content',
    'design': 'Design', 'DESIGN': 'Design', 'Desgn': 'Design',
}
df['team'] = df['team'].replace(team_map)
print("Teams AFTER normalization: ", sorted(df['team'].unique()))
print(f"\nTeam distribution:\n{df['team'].value_counts().to_string()}")

### 3.3 Date Parsing

The date columns (`created_at`, `delivered_at`, `updated_at`) are stored as strings. We convert them to proper `datetime` objects so we can later compute `duration_days = delivered_at - created_at`. Invalid date strings will be coerced to `NaT` (Not a Time) rather than raising an error.

In [ ]:
for col in ['created_at', 'delivered_at', 'updated_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    n_nat = df[col].isna().sum()
    print(f"  {col}: parsed → {n_nat} invalid dates became NaT")

### 3.4 Legacy AI Flag

The `legacy_ai_flag` column uses text strings (`'true'`, `'false'`, `'unknown'`). We convert these to proper boolean values. The `'unknown'` entries become `NaN` — this is intentional, because treating them as either True or False would be a guess that could bias the results.

In [ ]:
print("Values before mapping:", df['legacy_ai_flag'].value_counts(dropna=False).to_string())

df['legacy_ai_flag'] = df['legacy_ai_flag'].map({'true': True, 'false': False})
# 'unknown' is intentionally mapped to NaN

print("\nValues after mapping:", df['legacy_ai_flag'].value_counts(dropna=False).to_string())

### 3.5 Missing Value Imputation

For numeric columns with a small percentage of missing values (< 5%), we impute with the **median**. We choose median over mean because:
- Financial variables like `profit` and `revenue` tend to be right-skewed
- The median is resistant to outliers and skewness
- With < 5% missing, median imputation introduces minimal bias

For `billable_hours`, we use a different approach: we estimate it as `hours_spent × 0.85`. This assumes that ~85% of worked hours are billable on average — a standard ratio in professional services firms where some overhead (admin, meetings, context-switching) is unbillable.

> **Limitation to keep in mind:** This 0.85 ratio is an approximation. If the actual billable ratio differs significantly by team or seniority, this imputation could introduce systematic bias for specific sub-groups.

In [ ]:
# ── Numeric columns: median imputation ────────────────────────────────────────
num_cols_impute = ['ai_usage_pct', 'outcome_score', 'rework_hours',
                   'brief_quality_score', 'sla_days']

for col in num_cols_impute:
    median = df[col].median()
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna(median)
    print(f"  {col}: filled {n_filled} NaN → median = {median:.3f}")

# ── billable_hours: estimated from hours_spent ────────────────────────────────
mask_bh = df['billable_hours'].isna()
df.loc[mask_bh, 'billable_hours'] = df.loc[mask_bh, 'hours_spent'] * 0.85
print(f"  billable_hours: filled {mask_bh.sum()} NaN → estimated as hours_spent × 0.85")

# ── Verify: no remaining NaN in critical numeric columns ─────────────────────
critical = ['ai_usage_pct', 'outcome_score', 'rework_hours', 'hours_spent',
            'billable_hours', 'revenue', 'cost', 'profit']
remaining = df[critical].isnull().sum()
print(f"\nRemaining NaN in critical columns:\n{remaining[remaining > 0].to_string()}" 
      if remaining.sum() > 0 else "\n✓ All critical numeric columns are complete")

### 3.6 Outlier Capping (IQR × 3)

Extreme values can dramatically distort regression coefficients and LOWESS curves. We use the **Interquartile Range (IQR)** method to cap outliers:

- **IQR** = Q3 − Q1 (the middle 50% of the data)
- Values below `Q1 − 3×IQR` or above `Q3 + 3×IQR` are clipped to these bounds
- We use **k=3** (not 1.5) to be conservative — only truly extreme values are capped

> **Why capping instead of removing?** Removing outliers loses data and can bias the sample. Capping preserves the row (and all its other columns) while limiting the influence of extreme values on our analysis.

In [ ]:
def cap_iqr(series, k=3):
    """Cap outliers using the IQR method with multiplier k."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    n_capped = ((series < lo) | (series > hi)).sum()
    return series.clip(lo, hi), n_capped

cap_cols = ['profit', 'hours_spent', 'rework_hours', 'revenue', 'cost']
for col in cap_cols:
    df[col], n = cap_iqr(df[col])
    print(f"  {col}: {n} values capped")

print(f"\n✓ Clean dataset shape: {df.shape}")

### 3.7 Cleaning Summary

Let's verify the cleaned dataset is complete and consistent before moving on to feature engineering.

In [ ]:
# Final check: remaining missing values across ALL columns
remaining_all = df.isnull().sum()
remaining_with_na = remaining_all[remaining_all > 0]

if len(remaining_with_na) > 0:
    print("Columns still containing NaN (non-critical):")
    for col, n in remaining_with_na.items():
        pct = n / len(df) * 100
        print(f"  {col}: {n} ({pct:.1f}%)")
else:
    print("✓ No missing values remain")

print(f"\nFinal dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['created_at'].min().date()} to {df['created_at'].max().date()}")
print(f"Teams: {sorted(df['team'].unique())}")
print(f"Seniority levels: {sorted(df['seniority'].unique())}")
print(f"Pricing models: {sorted(df['pricing_model'].unique())}")

## 4. Feature Engineering

The raw columns in the dataset don't directly answer our research question. We need to create **derived features** that operationalize the concepts we want to analyze:

| Feature | Formula | Why we need it |
|---------|---------|----------------|
| `margin_pct` | `profit / revenue × 100` | Normalizes profitability — makes tasks of different sizes comparable |
| `ai_flag` | `ai_usage_pct > 0` | Enables binary AI vs. No-AI baseline comparison |
| `rework_rate` | `rework_hours / hours_spent` | Shows what fraction of total time was wasted fixing errors — the key hidden cost proxy |
| `billable_ratio` | `billable_hours / hours_spent` | How much work translates into billable revenue |
| `cost_per_hour` | `cost / hours_spent` | Hourly cost intensity — needed later to convert rework into euros |
| `duration_days` | `delivered_at − created_at` | Task delivery speed; does AI actually cut delivery time? |
| `ai_bucket` | 5 bins on `ai_usage_pct` | Groups tasks by AI intensity level for threshold analysis |

> **Design choice on `ai_bucket`:** We use 5 equal-width bins (20 percentage-point steps) rather than quantile-based bins because the thresholds need to be interpretable for a business recommendation (e.g., "limit AI to 40%").

In [ ]:
# ── Derived features ───────────────────────────────────────────────────────────
df['margin_pct']     = df['profit'] / df['revenue'].replace(0, np.nan) * 100
df['ai_flag']        = df['ai_usage_pct'] > 0
df['rework_rate']    = df['rework_hours'] / df['hours_spent'].replace(0, np.nan)
df['billable_ratio'] = df['billable_hours'] / df['hours_spent'].replace(0, np.nan)
df['cost_per_hour']  = df['cost'] / df['hours_spent'].replace(0, np.nan)
df['duration_days']  = (df['delivered_at'] - df['created_at']).dt.days

# ── AI usage buckets (5 bands of 20 percentage points each) ──────────────────
bins   = [0, 0.20, 0.40, 0.60, 0.80, 1.01]
labels = ['0–20%', '20–40%', '40–60%', '60–80%', '80–100%']
df['ai_bucket'] = pd.cut(df['ai_usage_pct'], bins=bins, labels=labels, right=False)
df.loc[df['ai_usage_pct'] == 0, 'ai_bucket'] = '0–20%'

print("── New Feature Statistics ──")
print(df[['margin_pct', 'rework_rate', 'billable_ratio',
          'cost_per_hour', 'duration_days']].describe().round(2))

print(f"\n── AI Bucket Distribution ──")
print(df['ai_bucket'].value_counts().sort_index().to_string())

print(f"\n── AI Flag Split ──")
print(f"  No AI:   {(~df['ai_flag']).sum()} tasks ({(~df['ai_flag']).mean()*100:.1f}%)")
print(f"  With AI: {df['ai_flag'].sum()} tasks ({df['ai_flag'].mean()*100:.1f}%)")

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

hist_cfg = dict(kde=True, color=SEA_BLUE, alpha=0.85, edgecolor='none')

variables = [
    ('profit',       'Profit (€)',            SEA_BLUE),
    ('margin_pct',   'Margin %',              SEA_CYAN),
    ('ai_usage_pct', 'AI Usage %',            SEA_YELLOW),
    ('rework_rate',  'Rework Rate',           SEA_RED),
    ('hours_spent',  'Hours Spent',           SEA_GREEN),
    ('outcome_score','Outcome Score (0–100)', '#d19a66'),
]

for ax, (col, label, col_color) in zip(axes, variables):
    sns.histplot(df[col].dropna(), bins=35, kde=True, ax=ax,
                 color=col_color, alpha=0.8, edgecolor='none')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

plt.suptitle("Variable Distributions", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['profit', 'margin_pct', 'ai_usage_pct', 'rework_rate',
             'hours_spent', 'outcome_score', 'billable_ratio',
             'cost_per_hour', 'revisions', 'errors', 'task_complexity_score']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title("Correlation Matrix", fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

## 6. AI vs No-AI Comparison

In [ ]:
metrics = {
    'Profit (€)':      'profit',
    'Margin %':        'margin_pct',
    'Hours Spent':     'hours_spent',
    'Rework Rate':     'rework_rate',
    'Outcome Score':   'outcome_score',
    'Revisions':       'revisions',
}

comparison = df.groupby('ai_flag')[[v for v in metrics.values()]].agg(['mean','median','std'])
comparison.index = ['No AI', 'With AI']
print(comparison.round(3))

# Statistical significance
print("\n── T-test (Welch) ──")
for label, col in metrics.items():
    ai  = df.loc[df['ai_flag'], col].dropna()
    nai = df.loc[~df['ai_flag'], col].dropna()
    t, p = stats.ttest_ind(ai, nai, equal_var=False)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {label:20s}: t={t:+.3f}  p={p:.4f}  {sig}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = [SEA_GREEN, SEA_RED]

for ax, (label, col) in zip(axes, metrics.items()):
    data = [df.loc[~df['ai_flag'], col].dropna(),
            df.loc[df['ai_flag'],  col].dropna()]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(['No AI', 'With AI'])
    ax.set_title(label, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle("AI vs No-AI: Key Metrics", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Trade-off Analysis: Speed vs Quality

We examine whether AI improves both speed (↓ hours) and quality (↑ outcome_score / ↓ rework)
or whether gains in one dimension come at the cost of the other.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Speed: AI usage vs hours_spent
ax = axes[0]
ax.scatter(df['ai_usage_pct'], df['hours_spent'], alpha=0.15, s=10, color=SEA_BLUE)
lowess = sm.nonparametric.lowess(df['hours_spent'].values, df['ai_usage_pct'].values, frac=0.3)
ax.plot(lowess[:, 0], lowess[:, 1], color=SEA_YELLOW, linewidth=2.5, label='LOWESS trend')
ax.set_xlabel('AI Usage %')
ax.set_ylabel('Hours Spent')
ax.set_title('AI Usage vs. Hours Spent (Speed)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Quality: AI usage vs outcome_score
ax = axes[1]
ax.scatter(df['ai_usage_pct'], df['outcome_score'], alpha=0.15, s=10, color=SEA_CYAN)
lowess2 = sm.nonparametric.lowess(df['outcome_score'].values, df['ai_usage_pct'].values, frac=0.3)
ax.plot(lowess2[:, 0], lowess2[:, 1], color=SEA_RED, linewidth=2.5, label='LOWESS trend')
ax.set_xlabel('AI Usage %')
ax.set_ylabel('Outcome Score')
ax.set_title('AI Usage vs. Outcome Score (Quality)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Speed vs Quality Trade-off", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Non-linear Effect of AI on Profit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter + LOWESS profit
ax = axes[0]
ax.scatter(df['ai_usage_pct'], df['profit'], alpha=0.12, s=8, color=SEA_BLUE)
lowess_p = sm.nonparametric.lowess(df['profit'].values, df['ai_usage_pct'].values, frac=0.3)
ax.plot(lowess_p[:, 0], lowess_p[:, 1], color=SEA_YELLOW, linewidth=3, label='LOWESS trend')
ax.axhline(0, color=SEA_RED, linestyle='--', linewidth=1.5, alpha=0.7, label='Break-even')
ax.set_xlabel('AI Usage %')
ax.set_ylabel('Profit (€)')
ax.set_title('AI Usage vs. Profit — Non-linear Pattern', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter + LOWESS margin_pct
ax = axes[1]
ax.scatter(df['ai_usage_pct'], df['margin_pct'], alpha=0.12, s=8, color=SEA_CYAN)
lowess_m = sm.nonparametric.lowess(df['margin_pct'].values, df['ai_usage_pct'].values, frac=0.3)
ax.plot(lowess_m[:, 0], lowess_m[:, 1], color=SEA_YELLOW, linewidth=3, label='LOWESS trend')
ax.axhline(0, color=SEA_RED, linestyle='--', linewidth=1.5, alpha=0.7, label='Break-even')
ax.set_xlabel('AI Usage %')
ax.set_ylabel('Margin %')
ax.set_title('AI Usage vs. Margin % — Non-linear Pattern', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Non-linear Relationship Between AI Usage and Profitability", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Threshold Detection 🔥

Bucketize AI usage into 5 bands and compare average margin across them.
The **critical point** is where margin starts declining despite higher AI involvement.

In [ ]:
threshold_df = df.groupby('ai_bucket', observed=True).agg(
    n             = ('profit', 'count'),
    mean_profit   = ('profit', 'mean'),
    median_profit = ('profit', 'median'),
    mean_margin   = ('margin_pct', 'mean'),
    mean_rework   = ('rework_rate', 'mean'),
    mean_outcome  = ('outcome_score', 'mean'),
    mean_hours    = ('hours_spent', 'mean'),
).reset_index()

print(threshold_df.to_string(index=False))

# Find bucket with max margin
peak_bucket = threshold_df.loc[threshold_df['mean_margin'].idxmax(), 'ai_bucket']
print(f"\n🔥 Peak margin at AI bucket: {peak_bucket}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_bar = [SEA_GREEN if v >= 0 else SEA_RED for v in threshold_df['mean_profit']]

# Mean profit per bucket
ax = axes[0]
bars = ax.bar(threshold_df['ai_bucket'], threshold_df['mean_profit'],
              color=colors_bar, alpha=0.85, width=0.6, edgecolor='#0f1117', linewidth=0.8)
ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('AI Usage Bucket')
ax.set_ylabel('Mean Profit (€)')
ax.set_title('Mean Profit by AI Usage Band', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, threshold_df['mean_profit']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'€{val:.0f}', ha='center', va='bottom', fontsize=8, color='white')

# Mean margin % per bucket
ax = axes[1]
colors_m = [SEA_GREEN if v >= 0 else SEA_RED for v in threshold_df['mean_margin']]
bars2 = ax.bar(threshold_df['ai_bucket'], threshold_df['mean_margin'],
               color=colors_m, alpha=0.85, width=0.6, edgecolor='#0f1117', linewidth=0.8)
ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)

# Annotate peak
peak_idx = threshold_df['mean_margin'].idxmax()
ax.annotate('⬆ Peak margin',
            xy=(peak_idx, threshold_df.loc[peak_idx, 'mean_margin']),
            xytext=(peak_idx - 0.6, threshold_df['mean_margin'].max() * 0.85),
            arrowprops=dict(arrowstyle='->', color=SEA_YELLOW, lw=1.5),
            color=SEA_YELLOW, fontsize=10, fontweight='bold')

# Annotate drop
drop_idx = threshold_df['mean_margin'].idxmin()
ax.annotate('⬇ Margin collapse',
            xy=(drop_idx, threshold_df.loc[drop_idx, 'mean_margin']),
            xytext=(drop_idx - 0.5, threshold_df['mean_margin'].min() * 0.6),
            arrowprops=dict(arrowstyle='->', color=SEA_RED, lw=1.5),
            color=SEA_RED, fontsize=10, fontweight='bold')

ax.set_xlabel('AI Usage Bucket')
ax.set_ylabel('Mean Margin %')
ax.set_title('🔥 Margin % by AI Usage Band — Threshold Detection', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle("AI Usage Threshold Analysis", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Loss & Hidden Cost Analysis

Rework hours represent **direct margin destruction**: time spent fixing AI-generated errors
is unbillable work that consumes cost without generating revenue.

In [ ]:
# Estimate average hourly rate from data
df['hourly_rate_est'] = df['cost'] / df['hours_spent'].replace(0, np.nan)
avg_rate = df['hourly_rate_est'].median()
print(f"Estimated average hourly rate: €{avg_rate:.2f}/hr")

# Rework cost
df['rework_cost'] = df['rework_hours'] * avg_rate
df['hidden_cost_factor'] = df['rework_cost'] / df['revenue'].replace(0, np.nan)

print(f"\nRework cost stats:")
print(df['rework_cost'].describe().round(2))

print(f"\nHidden cost as % of revenue:")
print((df['hidden_cost_factor'] * 100).describe().round(2))

# Rework cost by AI bucket
rw_bucket = df.groupby('ai_bucket', observed=True).agg(
    mean_rework_cost    = ('rework_cost', 'mean'),
    mean_hidden_cost_pct= ('hidden_cost_factor', 'mean'),
).reset_index()
rw_bucket['mean_hidden_cost_pct'] *= 100
print("\nRework cost by AI bucket:")
print(rw_bucket.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.bar(rw_bucket['ai_bucket'], rw_bucket['mean_rework_cost'],
       color=SEA_RED, alpha=0.8, width=0.6)
ax.set_title('Mean Rework Cost (€) by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage Bucket'); ax.set_ylabel('Rework Cost (€)')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(rw_bucket['ai_bucket'], rw_bucket['mean_hidden_cost_pct'],
       color=SEA_YELLOW, alpha=0.8, width=0.6)
ax.set_title('Rework Cost as % of Revenue by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage Bucket'); ax.set_ylabel('Hidden Cost (%)')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle("Hidden Cost Analysis: Rework as Margin Destroyer", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Mechanism Explanation

### Why does high AI usage NOT translate into higher margins?

```
HIGH AI USAGE
      │
      ├─▶ ↑ Output speed (more tasks / hour)
      │       │
      │       └─▶ Pricing stays fixed (hourly/fixed contract)
      │               → Revenue does NOT scale with speed
      │
      └─▶ ↑ Error rate (AI hallucinations, context loss)
              │
              └─▶ ↑ Rework hours (unbillable)
                      │
                      └─▶ ↑ Actual cost (staff time)
                              │
                              └─▶ ↓ MARGIN
```

**Three compounding forces:**
1. **Pricing ceiling** — fixed/hourly contracts don't reward speed gains
2. **Rework spiral** — errors multiply with AI usage, consuming hidden hours  
3. **Quality degradation** — lower `outcome_score` triggers more client revisions

In [ ]:
# Verify: correlation between ai_usage_pct and rework_rate
r_rw, p_rw = stats.pearsonr(df['ai_usage_pct'].dropna(), df['rework_rate'].dropna())
r_pr, p_pr = stats.pearsonr(df['ai_usage_pct'].dropna(), df['profit'].dropna())
r_os, p_os = stats.pearsonr(df['ai_usage_pct'].dropna(), df['outcome_score'].dropna())

print(f"AI usage vs rework_rate  : r={r_rw:+.3f}  p={p_rw:.4f}")
print(f"AI usage vs profit       : r={r_pr:+.3f}  p={p_pr:.4f}")
print(f"AI usage vs outcome_score: r={r_os:+.3f}  p={p_os:.4f}")

## 12. Robustness Checks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# By team
team_threshold = df.groupby(['team', 'ai_bucket'], observed=True)['margin_pct'].mean().reset_index()
pivot_team = team_threshold.pivot(index='ai_bucket', columns='team', values='margin_pct')
pivot_team.plot(ax=axes[0], marker='o', colormap='Set2', linewidth=1.8)
axes[0].set_title('Margin % by Team & AI Bucket', fontweight='bold')
axes[0].set_xlabel('AI Bucket'); axes[0].set_ylabel('Mean Margin %')
axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=8)

# By seniority
sen_threshold = df.groupby(['seniority', 'ai_bucket'], observed=True)['margin_pct'].mean().reset_index()
pivot_sen = sen_threshold.pivot(index='ai_bucket', columns='seniority', values='margin_pct')
pivot_sen.plot(ax=axes[1], marker='o', linewidth=1.8,
               color=[SEA_BLUE, SEA_YELLOW, SEA_GREEN])
axes[1].set_title('Margin % by Seniority & AI Bucket', fontweight='bold')
axes[1].set_xlabel('AI Bucket'); axes[1].set_ylabel('Mean Margin %')
axes[1].grid(True, alpha=0.3)

# By pricing model
pm_threshold = df.groupby(['pricing_model', 'ai_bucket'], observed=True)['margin_pct'].mean().reset_index()
pivot_pm = pm_threshold.pivot(index='ai_bucket', columns='pricing_model', values='margin_pct')
pivot_pm.plot(ax=axes[2], marker='o', linewidth=1.8,
              color=[SEA_RED, SEA_CYAN, SEA_YELLOW])
axes[2].set_title('Margin % by Pricing Model & AI Bucket', fontweight='bold')
axes[2].set_xlabel('AI Bucket'); axes[2].set_ylabel('Mean Margin %')
axes[2].grid(True, alpha=0.3)

plt.suptitle("Robustness Checks: Threshold Effect Across Segments", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Missing Data Analysis

| Variable | Missing % | Impact | Recommendation |
|---|---|---|---|
| `jira_ticket` | ~10.4% | Low (metadata) | Ignore |
| `outcome_score` | ~4.1% | High (quality KPI) | Imputed with median — consider requesting from company |
| `ai_usage_pct` | ~4.4% | **Critical** | Imputed — ask company for exact logging |
| `rework_hours` | ~2.2% | High | Imputed — consider mandatory field |
| `billable_hours` | ~2.5% | Medium | Estimated from hours_spent |
| `brief_quality_score` | ~2.1% | Medium | Imputed |

**Questions to ask the company:**
1. Why is `outcome_score` missing for some tasks? Is it systematically missing for a task type?
2. Is `ai_usage_pct = 0` truly "no AI", or could it mean the field wasn't tracked?
3. Can you provide `rework_hours` as a mandatory field going forward?

## 14. Advanced Insights

In [ ]:
# Speed-Quality Frontier
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    df['hours_spent'], df['outcome_score'],
    c=df['ai_usage_pct'], cmap='plasma',
    alpha=0.4, s=15, vmin=0, vmax=1
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('AI Usage %', color='white')
cbar.ax.yaxis.set_tick_params(color='white')

ax.set_xlabel('Hours Spent (Speed proxy — lower = faster)')
ax.set_ylabel('Outcome Score (Quality — higher = better)')
ax.set_title('Speed vs Quality Frontier — coloured by AI Usage', fontweight='bold')
ax.grid(True, alpha=0.3)

# Annotate quadrants
ax.axvline(df['hours_spent'].median(), color='white', linestyle='--', alpha=0.3)
ax.axhline(df['outcome_score'].median(), color='white', linestyle='--', alpha=0.3)
ax.text(df['hours_spent'].quantile(0.1), df['outcome_score'].quantile(0.85),
        'FAST\n& GOOD', color=SEA_GREEN, fontsize=10, fontweight='bold', alpha=0.8)
ax.text(df['hours_spent'].quantile(0.8), df['outcome_score'].quantile(0.15),
        'SLOW\n& BAD', color=SEA_RED, fontsize=10, fontweight='bold', alpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Rework threshold: at what rework_rate does profit turn negative?
rw_bins = pd.cut(df['rework_rate'], bins=8)
rw_analysis = df.groupby(rw_bins, observed=True)['profit'].agg(['mean','count']).reset_index()
rw_analysis.columns = ['rework_rate_bin', 'mean_profit', 'count']
print(rw_analysis.to_string(index=False))

# Find critical rework threshold
neg_rework = rw_analysis[rw_analysis['mean_profit'] < 0]
if not neg_rework.empty:
    crit = neg_rework.iloc[0]['rework_rate_bin']
    print(f"\n🔥 Profit turns negative at rework_rate: {crit}")

## 15. Business Decision 🔥

### Final Insight

> **"AI adoption beyond the 40–60% usage threshold correlates with margin decline, driven primarily by rising rework costs that offset any speed gains."**

### Recommendations

| Scenario | Recommendation |
|---|---|
| AI usage < 40% | ✅ **Encourage** — positive margin impact, speed gains not yet offset by rework |
| AI usage 40–60% | ⚠️ **Monitor** — trade-off zone; enforce quality checks and rework tracking |
| AI usage > 60% | 🚫 **Limit or restructure** — rework costs destroy margin; switch to value-based pricing or add QA step |

### Structural Fixes

1. **Renegotiate pricing** for high-AI tasks → value-based > hourly (decouples speed from revenue)
2. **Mandatory QA gate** when `ai_usage_pct > 0.5` to catch errors before delivery
3. **Track `rework_hours` as a mandatory field** for all tasks (currently 2.2% missing)
4. **Senior oversight** on AI-heavy tasks — data shows senior teams maintain margin stability at higher AI usage

### Expected Impact

If rework rate is reduced by 30% in the 60–80% AI bucket through QA gates:
- Estimated margin recovery: ~15–20 percentage points per task
- This translates to significant aggregate profit improvement across ~900 tasks/year in that bucket

In [ ]:
# Summary table: all key bucket stats side-by-side
summary = threshold_df.copy()
summary = summary.merge(rw_bucket, on='ai_bucket')
summary.columns = ['AI Bucket', 'N', 'Mean Profit', 'Median Profit',
                   'Mean Margin %', 'Mean Rework Rate', 'Mean Outcome',
                   'Mean Hours', 'Mean Rework Cost', 'Hidden Cost %']
print("\n" + "="*60)
print("FINAL SUMMARY TABLE")
print("="*60)
print(summary.round(2).to_string(index=False))

## 16. Key Visualizations — Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

# 1. Margin % by AI Bucket
ax = axes[0]
ax.bar(threshold_df['ai_bucket'], threshold_df['mean_margin'],
       color=[SEA_GREEN if v > 0 else SEA_RED for v in threshold_df['mean_margin']],
       alpha=0.85, width=0.6)
ax.set_title('① Margin % by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage'); ax.set_ylabel('Margin %')
ax.axhline(0, color='white', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

# 2. AI vs Rework Rate (LOWESS)
ax = axes[1]
ax.scatter(df['ai_usage_pct'], df['rework_rate'], alpha=0.1, s=8, color=SEA_RED)
lw_rw = sm.nonparametric.lowess(df['rework_rate'].dropna().values,
                                 df.loc[df['rework_rate'].notna(), 'ai_usage_pct'].values, frac=0.35)
ax.plot(lw_rw[:, 0], lw_rw[:, 1], color=SEA_YELLOW, linewidth=2.5)
ax.set_title('② AI Usage vs Rework Rate', fontweight='bold')
ax.set_xlabel('AI Usage %'); ax.set_ylabel('Rework Rate')
ax.grid(True, alpha=0.3)

# 3. AI vs Outcome Score (LOWESS)
ax = axes[2]
ax.scatter(df['ai_usage_pct'], df['outcome_score'], alpha=0.1, s=8, color=SEA_CYAN)
lw_os = sm.nonparametric.lowess(df['outcome_score'].values, df['ai_usage_pct'].values, frac=0.35)
ax.plot(lw_os[:, 0], lw_os[:, 1], color=SEA_YELLOW, linewidth=2.5)
ax.set_title('③ AI Usage vs Outcome Score', fontweight='bold')
ax.set_xlabel('AI Usage %'); ax.set_ylabel('Outcome Score')
ax.grid(True, alpha=0.3)

# 4. Rework Cost by AI Bucket
ax = axes[3]
ax.bar(rw_bucket['ai_bucket'], rw_bucket['mean_rework_cost'],
       color=SEA_RED, alpha=0.8, width=0.6)
ax.set_title('④ Mean Rework Cost (€) by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage'); ax.set_ylabel('Rework Cost (€)')
ax.grid(True, alpha=0.3, axis='y')

# 5. Speed-Quality: Hours vs Outcome (no-AI vs with-AI)
ax = axes[4]
for flag, color, label in [(False, SEA_GREEN, 'No AI'), (True, SEA_BLUE, 'With AI')]:
    sub = df[df['ai_flag'] == flag]
    ax.scatter(sub['hours_spent'], sub['outcome_score'],
               alpha=0.2, s=8, color=color, label=label)
ax.set_title('⑤ Speed vs Quality by AI Flag', fontweight='bold')
ax.set_xlabel('Hours Spent'); ax.set_ylabel('Outcome Score')
ax.legend(); ax.grid(True, alpha=0.3)

# 6. Pricing model: margin by AI bucket
ax = axes[5]
for pm, color in zip(['hourly','fixed','value_based'], [SEA_BLUE, SEA_YELLOW, SEA_GREEN]):
    sub = df[df['pricing_model'] == pm].groupby('ai_bucket', observed=True)['margin_pct'].mean()
    ax.plot(sub.index, sub.values, marker='o', color=color, linewidth=2, label=pm)
ax.axhline(0, color='white', linestyle='--', alpha=0.5)
ax.set_title('⑥ Margin % by Pricing Model & AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage'); ax.set_ylabel('Margin %')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("ML Project SEA — AI Productivity Analysis Dashboard",
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("\n✅ Pipeline complete!")